In [ ]:
import os
os.environ["PYTHONNOUSERSITE"] = "1"

import site
print(site.ENABLE_USER_SITE)
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from dotenv import load_dotenv 

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq


: 

In [ ]:
EMBEDDING_MODEL='sentence-transformers/all-MiniLM-L6-v2'
RE_RANKING_MODEL='cross-encoder/ms-marco-MiniLM-L-6-v2'
CHUNK_SIZE=400
CHUNK_OVERLAP=150
VECTOR_TOP_K=10
BM25_TOP_K=10
FINAL_TOP_K=10

PDF_DIRECTORY=os.getcwd()
INDEX_NAME='resume-rag'

NOTE_BOOK_PATH = os.getcwd()
DB_PATH = os.path.join(NOTE_BOOK_PATH, "website_chroma_db")

GROQ_API_KEY= userdata.get('GROQ_API_KEY')
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

if not GROQ_API_KEY:
  raise ValueError("GROQ_API_KEY not found in Colab secrets. Please set it.")

if not GOOGLE_API_KEY:
  raise ValueError("GOOGLE_API_KEY not found in Colab secrets. Please set it.")

In [ ]:
import re

def tokenizer(text):
  return re.findall(r'\w+', text.lower())

class SentenceTransformerEmbedding(Embeddings):
  def __init__(self , model_name):
    self.model=SentenceTransformer(model_name)

  def embed_documents(self , texts):
    return self.model.encode(texts , normalize_embeddings=True).tolist()

  def embed_query(self, text):
    return self.model.encode(text , normalize_embeddings=True).tolist()

In [ ]:

def load_doc(directory_path , extension=['pdf']):
  documents =[]
  for exe in extension:
    loader = DirectoryLoader(
        directory_path,
        glob=f"**/*.{exe}",
        show_progress=True,
        use_multithreading=True
    )
    documents.extend(loader.load())
  return documents


def chunk_documents(docs):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=CHUNK_SIZE,
      chunk_overlap=CHUNK_OVERLAP
  )
  return splitter.split_documents(docs)

In [ ]:
def building_indexes(pdf_path):
    docs = load_doc(pdf_path)
    chunks = chunk_documents(docs)

    print(f"Total chunks: {len(chunks)}")

    embeddings = SentenceTransformerEmbedding(EMBEDDING_MODEL)

    vector_store = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embeddings
    )

    if vector_store._collection.count() == 0:
        print("Creating new vector database...")
        vector_store.add_documents(chunks)
        print(f"Indexed {vector_store._collection.count()} chunks.")
    else:
        print(f"Loaded existing DB with {vector_store._collection.count()} vectors.")

    corpus = [doc.page_content for doc in chunks]

    if len(corpus) == 0:
        raise ValueError("No chunks available to build BM25 index.")

    bm25 = BM25Okapi([tokenizer(text) for text in corpus])

    return vector_store, bm25, chunks


def reciprocal_rank_fusion(result_list , k=60):
  scores ={}
  lookup={}
  for results in result_list:
    for rank , doc in enumerate(results):
      key= doc.page_content
      lookup[key] = doc
      scores[key] = scores.get(key,0)+1 /(k+rank+1)
  ranked = sorted(scores.items() , key=lambda x:x[1] , reverse=True)
  return [lookup[k] for k , _ in ranked]

In [ ]:


if __name__ =="__main__":
#   llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0.2,
#     api_key=GOOGLE_API_KEY
# )
  llm  = ChatGroq(
       model="llama-3.3-70b-versatile",
        temperature=0.9,
        api_key="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"
  )

  reranker = CrossEncoder(RE_RANKING_MODEL)
  vector_store , bm25 , chunks = building_indexes(PDF_DIRECTORY)

  def hybrid_search(query):

    vec = vector_store.similarity_search(query, k=VECTOR_TOP_K)
    print("Collection count:", vector_store._collection.count())

    scores = bm25.get_scores(tokenizer(query))
    idx = sorted(range(len(scores)) , key=lambda i: scores[i], reverse=True)[:BM25_TOP_K]
    kw = [chunks[i] for i in idx]
    return reciprocal_rank_fusion([vec , kw])



  def rerank(query , docs):
    pairs = [[query , d.page_content]  for d in docs]
    scores= reranker.predict(pairs)
    ranked = sorted(zip(scores , docs) , key=lambda x:x[0] , reverse=True)
    return [ doc for _ , doc in ranked[:FINAL_TOP_K]]

  def retrieve_context(query):
    docs = hybrid_search(query)
    docs = rerank(query , docs)
    return '\n\n'.join(doc.page_content for doc in docs)

  prompt = ChatPromptTemplate.from_template("""
  You are an enterprise AI assistant.
Instructions:
1. Answer ONLY using provided context.
2. Use markdown headings and bullet points.
3. Highlight keywords in **bold**.
4. If answer unavailable, say: I do not know.

Context:
{context}
Question:
{query}

  """)
  chain = (
    {
        "context": RunnablePassthrough() | retrieve_context,
        "query": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)
  while True:
    q=input("Quetion:")
    if q.lower() == 'exit':
      break

    response = chain.invoke(q)
    print(response)


In [ ]:
from pathlib import Path


BASE_DIR = Path(__file__).resolve().parent.parent
LOG_FILE = BASE_DIR / "monitoring" / "audit.json"
print(BASE_DIR , LOG_FILE)